In [1]:
# ============================================================
# STEP 3 - CELL 1
# SETUP
# ============================================================

from pathlib import Path

import pandas as pd
import numpy as np

import re
import html
import unicodedata
import gc
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 150)

INPUT_DIR = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/step3")

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 70)
print("STEP 3 - TEXT PREPROCESSING")
print("=" * 70)

print("Input :", INPUT_DIR)
print("Output:", OUTPUT_DIR)

print("\n✅ Setup completed")

STEP 3 - TEXT PREPROCESSING
Input : /kaggle/input
Output: /kaggle/working/step3

✅ Setup completed


In [2]:
# ============================================================
# STEP 3 - CELL 2
# DISCOVER & LOAD DATA
# ============================================================

def find_file(filename):

    matches = list(
        INPUT_DIR.rglob(filename)
    )

    if not matches:
        raise FileNotFoundError(
            f"{filename} not found"
        )

    return matches[0]


products_path = find_file("products.csv")
reviews_path = find_file("reviews.csv")
interactions_path = find_file("interactions.csv")
cf_path = find_file("cf_interactions.csv")


print("Products     :", products_path)
print("Reviews      :", reviews_path)
print("Interactions :", interactions_path)
print("CF           :", cf_path)


products = pd.read_csv(
    products_path,
    low_memory=False
)

reviews = pd.read_csv(
    reviews_path,
    low_memory=False
)

interactions = pd.read_csv(
    interactions_path,
    low_memory=False
)

cf_interactions = pd.read_csv(
    cf_path,
    low_memory=False
)


print("\n" + "=" * 70)
print("DATA LOADED")
print("=" * 70)

print("Products       :", products.shape)
print("Reviews        :", reviews.shape)
print("Interactions   :", interactions.shape)
print("CF interactions:", cf_interactions.shape)

Products     : /kaggle/input/datasets/trnquangtriu/3dataset/products.csv
Reviews      : /kaggle/input/datasets/trnquangtriu/3dataset/reviews.csv
Interactions : /kaggle/input/datasets/trnquangtriu/3dataset/interactions.csv
CF           : /kaggle/input/datasets/trnquangtriu/3dataset/cf_interactions.csv

DATA LOADED
Products       : (164926, 15)
Reviews        : (767179, 11)
Interactions   : (767179, 5)
CF interactions: (259427, 5)


# Audit text trước preprocessing

In [3]:
# ============================================================
# STEP 3 - CELL 3
# TEXT QUALITY AUDIT
# ============================================================

print("=" * 70)
print("TEXT QUALITY AUDIT")
print("=" * 70)


# ------------------------------------------------------------
# REVIEWS
# ------------------------------------------------------------

print("\n[1] REVIEW TEXT")
print("-" * 70)

review_text = (
    reviews["review_text"]
    .fillna("")
    .astype(str)
)

review_length = (
    review_text.str.len()
)

print("Total reviews :", f"{len(reviews):,}")
print("Missing       :", f"{reviews['review_text'].isna().sum():,}")
print("Empty         :", f"{review_text.str.strip().eq('').sum():,}")

print(
    "Length median :",
    round(review_length.median(), 2)
)

print(
    "Length mean   :",
    round(review_length.mean(), 2)
)

print(
    "Exact duplicate texts:",
    f"{review_text.duplicated().sum():,}"
)


print("\nReview text by source:")

for source, group in reviews.groupby("source"):

    text = (
        group["review_text"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    print(
        f"{source:<10} "
        f"rows={len(group):>8,} | "
        f"empty={text.eq('').sum():>7,} | "
        f"unique={text.nunique():>8,}"
    )


# ------------------------------------------------------------
# PRODUCTS
# ------------------------------------------------------------

print("\n[2] PRODUCT TEXT FIELDS")
print("-" * 70)

product_text_fields = [
    "product_name",
    "category",
    "subcategory",
    "brand",
    "description"
]

for field in product_text_fields:

    if field not in products.columns:
        continue

    valid = (
        products[field]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
    )

    print(
        f"{field:<20}: "
        f"{valid.sum():>8,} / "
        f"{len(products):,} "
        f"({valid.mean()*100:.2f}%)"
    )

TEXT QUALITY AUDIT

[1] REVIEW TEXT
----------------------------------------------------------------------
Total reviews : 767,179
Missing       : 403
Empty         : 403
Length median : 130.0
Length mean   : 262.56
Exact duplicate texts: 64,413

Review text by source:
amazon     rows= 736,093 | empty=    403 | unique= 672,029
shopee     rows=  31,086 | empty=      0 | unique=  30,745

[2] PRODUCT TEXT FIELDS
----------------------------------------------------------------------
product_name        :  164,925 / 164,926 (100.00%)
category            :  162,960 / 164,926 (98.81%)
subcategory         :    1,434 / 164,926 (0.87%)
brand               :  163,428 / 164,926 (99.09%)
description         :   41,575 / 164,926 (25.21%)


# Hàm làm sạch text

Ta làm light normalization:

Unicode NFC;

decode HTML entities;

bỏ HTML tags;

bỏ URL khỏi text;

chuẩn hóa whitespace;

giữ dấu tiếng Việt;

giữ chữ và số;

không stopword removal;

không stemming;

không dịch Anh ↔ Việt.

In [4]:
# ============================================================
# STEP 3 - CELL 4
# TEXT NORMALIZATION FUNCTIONS
# ============================================================

URL_PATTERN = re.compile(
    r"https?://\S+|www\.\S+",
    flags=re.IGNORECASE
)

HTML_TAG_PATTERN = re.compile(
    r"<[^>]+>"
)

WHITESPACE_PATTERN = re.compile(
    r"\s+"
)


def normalize_text(value):
    """
    Light normalization for multilingual semantic embeddings.
    Keeps Vietnamese accents and meaningful punctuation.
    """

    if pd.isna(value):
        return ""

    text = str(value)

    # HTML entities
    text = html.unescape(text)

    # Unicode normalization
    text = unicodedata.normalize(
        "NFC",
        text
    )

    # Remove URLs
    text = URL_PATTERN.sub(
        " ",
        text
    )

    # Remove HTML tags
    text = HTML_TAG_PATTERN.sub(
        " ",
        text
    )

    # Remove control characters
    text = "".join(
        char if (
            char.isprintable()
            or char in "\n\t"
        )
        else " "
        for char in text
    )

    # Normalize whitespace
    text = WHITESPACE_PATTERN.sub(
        " ",
        text
    )

    return text.strip()


# ------------------------------------------------------------
# Quick test
# ------------------------------------------------------------

examples = [
    "<p>Sản phẩm rất tốt!</p>",
    "Xem tại https://example.com sản phẩm này",
    "  Very   good   product!  ",
    "Tai nghe Bluetooth chất lượng tốt 👍"
]

for x in examples:

    print("RAW  :", x)
    print("CLEAN:", normalize_text(x))
    print()

RAW  : <p>Sản phẩm rất tốt!</p>
CLEAN: Sản phẩm rất tốt!

RAW  : Xem tại https://example.com sản phẩm này
CLEAN: Xem tại sản phẩm này

RAW  :   Very   good   product!  
CLEAN: Very good product!

RAW  : Tai nghe Bluetooth chất lượng tốt 👍
CLEAN: Tai nghe Bluetooth chất lượng tốt 👍



# PREPROCESS REVIEW TEXT


In [5]:
# ============================================================
# STEP 3 - CELL 5
# PREPROCESS REVIEW TEXT
# ============================================================

print("=" * 70)
print("PREPROCESS REVIEW TEXT")
print("=" * 70)

reviews_clean = reviews.copy()

reviews_clean["clean_review"] = (
    reviews_clean["review_text"]
    .apply(normalize_text)
)

reviews_clean["char_count"] = (
    reviews_clean["clean_review"]
    .str.len()
)

reviews_clean["word_count"] = (
    reviews_clean["clean_review"]
    .str.split()
    .str.len()
)


print("\n[1] CLEANING RESULT")
print("-" * 70)

print(
    "Reviews:",
    f"{len(reviews_clean):,}"
)

print(
    "Empty after cleaning:",
    f"{reviews_clean['clean_review'].eq('').sum():,}"
)

print(
    "Median words:",
    reviews_clean["word_count"].median()
)

print(
    "Mean words:",
    round(
        reviews_clean["word_count"].mean(),
        2
    )
)


print("\n[2] WORD COUNT BY SOURCE")
print("-" * 70)

display(
    reviews_clean
    .groupby("source")["word_count"]
    .describe(
        percentiles=[
            .25,
            .50,
            .75,
            .90,
            .95,
            .99
        ]
    )
    .round(2)
)

PREPROCESS REVIEW TEXT

[1] CLEANING RESULT
----------------------------------------------------------------------
Reviews: 767,179
Empty after cleaning: 413
Median words: 25.0
Mean words: 48.55

[2] WORD COUNT BY SOURCE
----------------------------------------------------------------------


,count,mean,std,min,25%,50%,75%,90%,95%,99%,max
source,,,,,,,,,,,
amazon,736093.0,49.38,82.27,0.0,9.0,25.0,57.0,114.0,173.0,384.0,3960.0
shopee,31086.0,28.82,17.66,1.0,18.0,26.0,34.0,47.0,59.0,88.0,454.0


# Review validity

In [6]:
# ============================================================
# STEP 3 - CELL 6
# REVIEW SEMANTIC USABILITY
# ============================================================

print("=" * 70)
print("REVIEW SEMANTIC USABILITY")
print("=" * 70)

MIN_REVIEW_WORDS = 3

reviews_clean["semantic_usable"] = (
    reviews_clean["clean_review"].ne("")
    &
    (
        reviews_clean["word_count"]
        >= MIN_REVIEW_WORDS
    )
)


summary = (
    reviews_clean
    .groupby("source")
    .agg(
        total_reviews=(
            "review_id",
            "size"
        ),
        usable_reviews=(
            "semantic_usable",
            "sum"
        )
    )
)

summary["usable_%"] = (
    summary["usable_reviews"]
    / summary["total_reviews"]
    * 100
).round(2)

display(summary)


print(
    "\nTotal semantic usable:",
    f"{reviews_clean['semantic_usable'].sum():,}"
)

print(
    "Excluded from semantic embedding:",
    f"{(~reviews_clean['semantic_usable']).sum():,}"
)

REVIEW SEMANTIC USABILITY


,total_reviews,usable_reviews,usable_%
source,,,
amazon,736093,683832,92.90
shopee,31086,30966,99.61



Total semantic usable: 714,798
Excluded from semantic embedding: 52,381


# DUPLICATE REVIEW TEXT AUDIT


In [7]:
# ============================================================
# STEP 3 - CELL 7
# DUPLICATE REVIEW TEXT AUDIT
# ============================================================

print("=" * 70)
print("DUPLICATE REVIEW TEXT AUDIT")
print("=" * 70)

usable_reviews = (
    reviews_clean[
        reviews_clean["semantic_usable"]
    ]
)

text_counts = (
    usable_reviews["clean_review"]
    .value_counts()
)

duplicate_texts = (
    text_counts > 1
)

print(
    "Usable reviews:",
    f"{len(usable_reviews):,}"
)

print(
    "Unique texts:",
    f"{usable_reviews['clean_review'].nunique():,}"
)

print(
    "Texts appearing >1 time:",
    f"{duplicate_texts.sum():,}"
)

print(
    "Rows belonging to repeated texts:",
    f"{text_counts[duplicate_texts].sum():,}"
)


print("\nMost frequent repeated review texts:")

display(
    text_counts
    .head(20)
    .rename("count")
    .reset_index()
    .rename(
        columns={
            "clean_review":
                "review_text"
        }
    )
)

DUPLICATE REVIEW TEXT AUDIT
Usable reviews: 714,798
Unique texts: 693,264
Texts appearing >1 time: 8,005
Rows belonging to repeated texts: 29,539

Most frequent repeated review texts:


,review_text,count
0,Works as expected,274
1,Works as expected.,239
2,Easy to use,184
3,Works as advertised.,178
4,Just what I needed,168
5,Works as advertised,158
6,Easy to install,156
7,I love it,135
8,Does the job,131
9,works as expected,124


# PREPROCESS PRODUCT TEXT FIELDS


In [8]:
# ============================================================
# STEP 3 - CELL 8
# PREPROCESS PRODUCT TEXT FIELDS
# ============================================================

print("=" * 70)
print("PREPROCESS PRODUCT TEXT")
print("=" * 70)

products_clean = products.copy()

fields_to_clean = [
    "product_name",
    "category",
    "subcategory",
    "brand",
    "description"
]

for field in fields_to_clean:

    if field in products_clean.columns:

        products_clean[
            f"clean_{field}"
        ] = (
            products_clean[field]
            .apply(normalize_text)
        )


print("Created:")

for field in fields_to_clean:

    clean_field = f"clean_{field}"

    if clean_field in products_clean.columns:

        print(
            f"  {clean_field}"
        )

PREPROCESS PRODUCT TEXT
Created:
  clean_product_name
  clean_category
  clean_subcategory
  clean_brand
  clean_description


# BUILD PRODUCT SEMANTIC TEXT


In [9]:
# ============================================================
# STEP 3 - CELL 9
# BUILD PRODUCT SEMANTIC TEXT
# ============================================================

print("=" * 70)
print("BUILD PRODUCT SEMANTIC TEXT")
print("=" * 70)


def build_product_text(row):

    parts = []

    # Product name
    name = row.get(
        "clean_product_name",
        ""
    )

    if name:
        parts.append(
            f"Product: {name}"
        )

    # Category
    category = row.get(
        "clean_category",
        ""
    )

    if category:
        parts.append(
            f"Category: {category}"
        )

    # Subcategory
    subcategory = row.get(
        "clean_subcategory",
        ""
    )

    if subcategory:
        parts.append(
            f"Subcategory: {subcategory}"
        )

    # Brand
    brand = row.get(
        "clean_brand",
        ""
    )

    if brand:
        parts.append(
            f"Brand: {brand}"
        )

    # Description
    description = row.get(
        "clean_description",
        ""
    )

    if description:

        # Limit extremely long descriptions.
        # Character limit is only to control embedding input size.
        description = description[:2000]

        parts.append(
            f"Description: {description}"
        )

    return " | ".join(parts)


products_clean["product_text"] = (
    products_clean.apply(
        build_product_text,
        axis=1
    )
)

products_clean["product_text_word_count"] = (
    products_clean["product_text"]
    .str.split()
    .str.len()
)


print(
    "Products:",
    f"{len(products_clean):,}"
)

print(
    "Empty product_text:",
    f"{products_clean['product_text'].eq('').sum():,}"
)


print("\nProduct text length by source:")

display(
    products_clean
    .groupby("source")[
        "product_text_word_count"
    ]
    .describe(
        percentiles=[
            .25,
            .50,
            .75,
            .90,
            .95,
            .99
        ]
    )
    .round(2)
)

BUILD PRODUCT SEMANTIC TEXT
Products: 164,926
Empty product_text: 0

Product text length by source:


,count,mean,std,min,25%,50%,75%,90%,95%,99%,max
source,,,,,,,,,,,
amazon,121917.0,29.27,8.79,6.0,22.0,30.0,36.0,40.0,42.0,47.0,342.0
shopee,1434.0,25.19,5.65,11.0,21.0,25.0,29.0,32.0,34.0,39.0,53.0
tiki,41575.0,63.58,11.69,12.0,59.0,65.0,70.0,76.0,79.0,87.0,127.0


# PRODUCT SEMANTIC COVERAGE


In [10]:
# ============================================================
# STEP 3 - CELL 10
# PRODUCT SEMANTIC COVERAGE
# ============================================================

print("=" * 70)
print("PRODUCT SEMANTIC COVERAGE")
print("=" * 70)

products_clean["semantic_usable"] = (
    products_clean["product_text"]
    .str.strip()
    .ne("")
)


semantic_product_summary = (
    products_clean
    .groupby("source")
    .agg(
        total_products=(
            "product_id",
            "size"
        ),
        semantic_products=(
            "semantic_usable",
            "sum"
        )
    )
)

semantic_product_summary[
    "coverage_%"
] = (
    semantic_product_summary[
        "semantic_products"
    ]
    /
    semantic_product_summary[
        "total_products"
    ]
    * 100
).round(2)

display(
    semantic_product_summary
)


print("\nSample product semantic documents:")

sample_cols = [
    "product_id",
    "source",
    "product_name",
    "price",
    "currency",
    "product_text"
]

display(
    products_clean[
        sample_cols
    ]
    .sample(
        min(
            10,
            len(products_clean)
        ),
        random_state=42
    )
)

PRODUCT SEMANTIC COVERAGE


,total_products,semantic_products,coverage_%
source,,,
amazon,121917,121917,100.0
shopee,1434,1434,100.0
tiki,41575,41575,100.0



Sample product semantic documents:


,product_id,source,product_name,price,currency,product_text
47747,amazon_B072MLBMBF,amazon,"mCover iPearl Hard Shell Case for 2017 11.6"" Dell Chromebook 11 3189 Series 2-in-1 Laptop (NOT Compatible with 210-ACDU / 3120/3180 Series) - 3189...",19.99,USD,"Product: mCover iPearl Hard Shell Case for 2017 11.6"" Dell Chromebook 11 3189 Series 2-in-1 Laptop (NOT Compatible with 210-ACDU / 3120/3180 Serie..."
124900,tiki_124778123,tiki,Dép tổ ong Vàng thái loại xịn cho nam,34999.00,VND,Product: Dép tổ ong Vàng thái loại xịn cho nam | Category: Dép nam quai ngang | Brand: OEM | Description: Dép tổ ong là loại dép ai cũng biết rồi ...
126307,tiki_44324447,tiki,Túi xách da bò công sở cao cấp phong cách châu âu TX DA BÒ 7612,801000.00,VND,Product: Túi xách da bò công sở cao cấp phong cách châu âu TX DA BÒ 7612 | Category: Thời Trang | Brand: ANANSHOP688 | Description: Kích thước : r...
95258,amazon_B07C2C7ZSB,amazon,"Mission Darkness NeoLok Non-Window Faraday Bag for Tablets (+ Secure Magnetic Closure) // Device Shielding for Law Enforcement & Military, Data Se...",65.00,USD,Product: Mission Darkness NeoLok Non-Window Faraday Bag for Tablets (+ Secure Magnetic Closure) // Device Shielding for Law Enforcement & Military...
14211,amazon_B08HCLB1JZ,amazon,"LP-E10 Battery Charger Pack, LP 2-Pack Battery & Charger, Compatible with Canon EOS Rebel T7, T6, T5, T3, T100, 4000D, 3000D, 2000D, 1500D, 1300D,...",26.98,USD,"Product: LP-E10 Battery Charger Pack, LP 2-Pack Battery & Charger, Compatible with Canon EOS Rebel T7, T6, T5, T3, T100, 4000D, 3000D, 2000D, 1500..."
10003,amazon_B07VBKH9NH,amazon,"Messenger Bag for Men,VASCHY Vintage Water Resistant Canvas Satchel 14 15.6 17inch Laptop Briefcase",46.99,USD,"Product: Messenger Bag for Men,VASCHY Vintage Water Resistant Canvas Satchel 14 15.6 17inch Laptop Briefcase | Category: AMAZON FASHION | Brand: V..."
6059,amazon_B07RT577T7,amazon,9inch Truck GPS Big Touchscreen Trucking GPS XGODY GPS Navigation for car Navigation 8GB ROM SAT NAV System Navigator Driving Alarm Lifetime Map U...,92.99,USD,Product: 9inch Truck GPS Big Touchscreen Trucking GPS XGODY GPS Navigation for car Navigation 8GB ROM SAT NAV System Navigator Driving Alarm Lifet...
127853,tiki_26210394,tiki,Hình Xăm Dán Đầu Lâu QSA025,50000.00,VND,Product: Hình Xăm Dán Đầu Lâu QSA025 | Category: Thời Trang | Brand: OEM | Description: Thông tin sản phẩm:Hình xăm dán là sản phẩm giúp nổi bật c...
135264,tiki_36756069,tiki,TÚI TOTE ĐEO CHÉO CHÚ VỊT OM0059,107000.00,VND,Product: TÚI TOTE ĐEO CHÉO CHÚ VỊT OM0059 | Category: Root | Brand: OEM | Description: Túi tote là một loại túi lớn có dây xách song song nhau nối...
111570,amazon_B00563LXDY,amazon,"Intel Socket 775 Copper Core/Aluminum Heat Sink & 3.5"" Ball Bearing Fan w/4-Pin Connector up to Core 2 Quad 3.0GHz",29.97,USD,"Product: Intel Socket 775 Copper Core/Aluminum Heat Sink & 3.5"" Ball Bearing Fan w/4-Pin Connector up to Core 2 Quad 3.0GHz | Category: Computers ..."


# BUILD REVIEW RAG DOCUMENT


In [11]:
# ============================================================
# STEP 3 - CELL 11
# BUILD REVIEW RAG DOCUMENT
# ============================================================

print("=" * 70)
print("BUILD REVIEW RAG DOCUMENT")
print("=" * 70)

product_context = (
    products_clean[
        [
            "product_id",
            "product_name",
            "category",
            "brand"
        ]
    ]
    .drop_duplicates(
        "product_id"
    )
)

reviews_clean = (
    reviews_clean
    .merge(
        product_context,
        on="product_id",
        how="left"
    )
)


def build_review_document(row):

    parts = []

    product_name = row.get(
        "product_name",
        ""
    )

    if pd.notna(product_name) and str(product_name).strip():
        parts.append(
            f"Product: {product_name}"
        )

    category = row.get(
        "category",
        ""
    )

    if pd.notna(category) and str(category).strip():
        parts.append(
            f"Category: {category}"
        )

    brand = row.get(
        "brand",
        ""
    )

    if pd.notna(brand) and str(brand).strip():
        parts.append(
            f"Brand: {brand}"
        )

    rating = row.get(
        "rating",
        None
    )

    if pd.notna(rating):
        parts.append(
            f"Rating: {rating}/5"
        )

    review = row.get(
        "clean_review",
        ""
    )

    if review:
        parts.append(
            f"Review: {review}"
        )

    return " | ".join(parts)


reviews_clean["rag_document"] = (
    reviews_clean.apply(
        build_review_document,
        axis=1
    )
)


print(
    "RAG documents:",
    f"{reviews_clean['rag_document'].ne('').sum():,}"
)

print("\nSample:")

display(
    reviews_clean[
        [
            "review_id",
            "product_id",
            "language",
            "rag_document"
        ]
    ]
    .head(10)
)

BUILD REVIEW RAG DOCUMENT
RAG documents: 767,179

Sample:


,review_id,product_id,language,rag_document
0,review_0000000,amazon_B07N69T6TM,en,"Product: Toys for 4-5 Year Old Boys, Mom&myaboys 8 X 21 Kids Binoculars for Children,Compact Telescope Boys Gifts 4-8 Years Old to Bird Watching &..."
1,review_0000001,amazon_B01G8JO5F2,en,"Product: Senso Bluetooth Headphones, Best Wireless Sports Earbuds w/Mic IPX7 Waterproof HD Stereo Sweatproof Earphones for Gym Running Workout Noi..."
2,review_0000002,amazon_B07CJYMRWM,en,Product: Edifier P841 Comfortable Noise Isolating Over-Ear Headphones with Microphone and Volume Controls - White | Category: Cell Phones & Access...
3,review_0000003,amazon_B07CML419K,en,"Product: Fancii Small Personal Desk USB Fan, Portable Mini Table Fan with Twin Turbo Blades, Whisper Quiet Cyclone Air Technology - For Home, Offi..."
4,review_0000004,amazon_B09S6Y5BRG,en,"Product: Otium Bluetooth Earbuds Wireless Headphones Bluetooth Headphones, Sports Earbuds, IPX7 Waterproof Stereo Earphones for Gym Running 15 Hou..."
5,review_0000005,amazon_B017T99JPG,en,Product: Bose SoundLink Mini Bluetooth Speaker II (Carbon) | Category: Home Audio & Theater | Brand: Bose | Rating: 5.0/5 | Review: yes.. so good....
6,review_0000006,amazon_B00L0YLRUW,en,Product: NETGEAR Wi-Fi Range Extender EX2700 - Coverage Up to 800 Sq Ft and 10 devices with N300 Wireless Signal Booster & Repeater (Up to 300Mbps...
7,review_0000007,amazon_B00IXNUDG2,en,"Product: Pack of 5 8x10 White Picture Mat, for 2 4x6 Photos or Pictures | Category: Arts, Crafts & Sewing | Brand: bux1 picture matting | Rating: ..."
8,review_0000008,amazon_B00PVCXB4Y,en,Product: D-Link Wi-Fi AC750 Dual Band Range Extender (DAP-1520) (Discontinued by Manufacturer) | Category: Computers | Brand: D-Link | Rating: 5.0...
9,review_0000009,amazon_B00X5C22SS,en,Product: Sony BDPS3500 Blu-ray Player with Wi-Fi (2015 Model) | Category: All Electronics | Brand: Sony | Rating: 5.0/5 | Review: This is the 3rd ...


# FINAL PREPROCESSING AUDIT


In [12]:
# ============================================================
# STEP 3 - CELL 12
# FINAL PREPROCESSING AUDIT
# ============================================================

print("=" * 70)
print("FINAL TEXT PREPROCESSING AUDIT")
print("=" * 70)

checks = {

    "product_rows_preserved":
        len(products_clean)
        == len(products),

    "review_rows_preserved":
        len(reviews_clean)
        == len(reviews),

    "product_ids_unique":
        products_clean[
            "product_id"
        ].is_unique,

    "review_ids_unique":
        reviews_clean[
            "review_id"
        ].is_unique,

    "product_text_exists":
        "product_text"
        in products_clean.columns,

    "clean_review_exists":
        "clean_review"
        in reviews_clean.columns,

    "rag_document_exists":
        "rag_document"
        in reviews_clean.columns,

    "cf_rows_preserved":
        len(cf_interactions) > 0
}


for name, status in checks.items():

    icon = (
        "✅"
        if status
        else "❌"
    )

    print(
        f"{icon} {name:<30}: {status}"
    )


print("\n" + "-" * 70)

print(
    "Products:",
    f"{len(products_clean):,}"
)

print(
    "Semantic products:",
    f"{products_clean['semantic_usable'].sum():,}"
)

print(
    "Reviews:",
    f"{len(reviews_clean):,}"
)

print(
    "Semantic reviews:",
    f"{reviews_clean['semantic_usable'].sum():,}"
)

print(
    "CF interactions:",
    f"{len(cf_interactions):,}"
)

assert all(checks.values())

print(
    "\n✅ Preprocessing consistency checks passed"
)

FINAL TEXT PREPROCESSING AUDIT
✅ product_rows_preserved        : True
✅ review_rows_preserved         : True
✅ product_ids_unique            : True
✅ review_ids_unique             : True
✅ product_text_exists           : True
✅ clean_review_exists           : True
✅ rag_document_exists           : True
✅ cf_rows_preserved             : True

----------------------------------------------------------------------
Products: 164,926
Semantic products: 164,926
Reviews: 767,179
Semantic reviews: 714,798
CF interactions: 259,427

✅ Preprocessing consistency checks passed


# SAVE STEP 3 OUTPUTS


In [13]:
# ============================================================
# STEP 3 - CELL 13
# SAVE STEP 3 OUTPUTS
# ============================================================

print("=" * 70)
print("SAVE STEP 3 OUTPUTS")
print("=" * 70)


# ------------------------------------------------------------
# Semantic review dataset
# ------------------------------------------------------------

semantic_reviews = (
    reviews_clean[
        reviews_clean[
            "semantic_usable"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Semantic product dataset
# ------------------------------------------------------------

semantic_products = (
    products_clean[
        products_clean[
            "semantic_usable"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

clean_reviews_path = (
    OUTPUT_DIR /
    "clean_reviews.csv"
)

clean_products_path = (
    OUTPUT_DIR /
    "clean_products.csv"
)

semantic_reviews_path = (
    OUTPUT_DIR /
    "semantic_reviews.csv"
)

semantic_products_path = (
    OUTPUT_DIR /
    "semantic_products.csv"
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

reviews_clean.to_csv(
    clean_reviews_path,
    index=False
)

products_clean.to_csv(
    clean_products_path,
    index=False
)

semantic_reviews.to_csv(
    semantic_reviews_path,
    index=False
)

semantic_products.to_csv(
    semantic_products_path,
    index=False
)


print("\nSaved:")
print("-" * 70)

for path in [
    clean_reviews_path,
    clean_products_path,
    semantic_reviews_path,
    semantic_products_path
]:

    size_mb = (
        path.stat().st_size /
        1024**2
    )

    print(
        f"{path.name:<30}"
        f"{size_mb:>10.2f} MB"
    )


print("\n" + "=" * 70)
print("✅ STEP 3 COMPLETED")
print("=" * 70)

SAVE STEP 3 OUTPUTS

Saved:
----------------------------------------------------------------------
clean_reviews.csv                 954.05 MB
clean_products.csv                111.27 MB
semantic_reviews.csv              927.83 MB
semantic_products.csv             111.27 MB

✅ STEP 3 COMPLETED
